<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/GEMINI4_CARBON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CASE1

In [ ]:
import re
from pathlib import Path
from google import genai
from google.genai import types
from google.colab import userdata

api_key = userdata.get('GEMINI')
client = genai.Client(api_key=api_key)

config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=2048),
    tools=[{"code_execution": {}}],
    max_output_tokens=65536,
)

prompt = """
Write a deterministic Python implementation of an asynchronous token-bucket
rate limiter with burst support and thread-safety locks.

Include unit tests verifying that concurrent requests respect the burst ceiling
and smoothly refill over time. Run the script to verify correctness.
"""

print("[*] Generating and verifying solution via gemini-4-carbon...")
response = client.models.generate_content(
    model="gemini-4-carbon",
    contents=prompt,
    config=config,
)

# 1. Verify that sandbox execution succeeded
sandbox_passed = any(
    getattr(part, "code_execution_result", None) and part.code_execution_result.outcome == "OUTCOME_OK"
    for part in response.candidates[0].content.parts
)

# 2. Extract the complete, self-contained Python program from the synthesized markdown text
full_text = response.text or ""
code_blocks = re.findall(r"```python\s*(.*?)\s*```", full_text, flags=re.DOTALL)

if code_blocks:
    # Use the largest code block (guaranteed to be the full implementation, not a partial snippet)
    complete_code = max(code_blocks, key=len)
else:
    # Fallback to concatenating executable parts if text extraction fails
    code_parts = [
        part.executable_code.code
        for part in response.candidates[0].content.parts
        if getattr(part, "executable_code", None)
    ]
    complete_code = "\n\n".join(code_parts)

output_path = Path("async_token_bucket.py")
output_path.write_text(complete_code, encoding="utf-8")

status = "VERIFIED (Sandbox OUTCOME_OK)" if sandbox_passed else "EXTRACTED"
print(f"[+] Status: {status}")
print(f"[+] Successfully wrote complete script to: {output_path.resolve()}")

# 3. Execute locally
print("\n[*] Executing full script in Colab runtime:\n" + "=" * 50)
!python3 async_token_bucket.py

[*] Generating and verifying solution via gemini-4-carbon...
[+] Status: VERIFIED (Sandbox OUTCOME_OK)
[+] Successfully wrote complete script to: /content/async_token_bucket.py

[*] Executing full script in Colab runtime:
test_deterministic_burst_and_concurrent_refill (__main__.TestAsyncTokenBucket.test_deterministic_burst_and_concurrent_refill)
Uses a deterministic virtual clock to verify exact burst capping and ... ok
test_input_validation (__main__.TestAsyncTokenBucket.test_input_validation) ... ok
test_multithreaded_contention_across_event_loops (__main__.TestAsyncTokenBucket.test_multithreaded_contention_across_event_loops)
Verifies thread-safety locks by hammering a single AsyncTokenBucket ... Executing <Task finished name='Task-22' coro=<TestAsyncTokenBucket.test_multithreaded_contention_across_event_loops() done, defined at /content/async_token_bucket.py:212> result=None created at /usr/lib/python3.13/asyncio/runners.py:101> took 0.124 seconds
ok
test_real_wallclock_concurrent_

## CASE2

In [4]:
import re
from pathlib import Path
from google import genai
from google.genai import types
from google.colab import userdata

api_key = userdata.get('GEMINI')
client = genai.Client(api_key=api_key)

config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=2048),
    tools=[{"code_execution": {}}],
    max_output_tokens=65536,
)

prompt = """
Write an updated, high-performance Python implementation of an asynchronous
token-bucket rate limiter designed for extreme concurrency.

Include a rigorous benchmarking test suite that verifies:
1. Massive concurrent load (at least 5,000 simultaneous coroutines hammering the limiter).
2. Large token-bucket capacities (e.g., burst ceiling of 1,000 tokens with rapid refill rates).
3. Precise latency, drop rate, and throughput measurements under pressure.

Run the script to verify correctness and print out the benchmark summary metrics.
"""

print("[*] Generating high-concurrency benchmark script via gemini-4-carbon...")
response = client.models.generate_content(
    model="gemini-4-carbon",
    contents=prompt,
    config=config,
)

# 1. Verify that sandbox execution succeeded
sandbox_passed = any(
    getattr(part, "code_execution_result", None) and part.code_execution_result.outcome == "OUTCOME_OK"
    for part in response.candidates[0].content.parts
)

# 2. Extract the complete, self-contained Python program from the synthesized markdown text
full_text = response.text or ""
code_blocks = re.findall(r"```python\s*(.*?)\s*```", full_text, flags=re.DOTALL)

if code_blocks:
    complete_code = max(code_blocks, key=len)
else:
    code_parts = [
        part.executable_code.code
        for part in response.candidates[0].content.parts
        if getattr(part, "executable_code", None)
    ]
    complete_code = "\n\n".join(code_parts)

output_path = Path("async_token_bucket_benchmark.py")
output_path.write_text(complete_code, encoding="utf-8")

status = "VERIFIED (Sandbox OUTCOME_OK)" if sandbox_passed else "EXTRACTED"
print(f"[+] Status: {status}")
print(f"[+] Successfully wrote benchmark script to: {output_path.resolve()}")

# 3. Execute locally
print("\n[*] Executing benchmark script in Colab runtime:\n" + "=" * 50)
!python3 async_token_bucket_benchmark.py

[*] Generating high-concurrency benchmark script via gemini-4-carbon...
[+] Status: VERIFIED (Sandbox OUTCOME_OK)
[+] Successfully wrote benchmark script to: /content/async_token_bucket_benchmark.py

[*] Executing benchmark script in Colab runtime:
HIGH-CONCURRENCY ASYNC TOKEN BUCKET — BENCHMARK & VERIFICATION SUITE

[OK] Invariant checks passed (burst ceiling, non-blocking fast-path, cancellation debt refund).

┌── Test 1: Unbounded Queueing Under Herd (6,000 coroutines, Burst=1,000, Rate=5,000/s)
│ Concurrency       : 6,000 simultaneous coroutines
│ Requests          : 6,000 total | 6,000 acquired | 0 dropped (0.00%)
│ Wall Time         : 1.0159 s
│ Effective Rate    : 5,906.2 tokens/sec
│ Acquired Latency  : p50=384.304ms | p95=904.430ms | p99=952.002ms | max=963.089ms
└──────────────────────────────────────────────────────────────────────────────

┌── Test 2: SLA Load-Shedding (10,000 coroutines, max_wait=500ms, Burst=1,000, Rate=4,000/s)
│ Concurrency       : 10,000 simultaneous c

## CASE3

In [2]:
import re
import logging
from pathlib import Path
from google import genai
from google.genai import types
from google.colab import userdata

# Suppress non-text part warnings from the Google GenAI SDK
logging.getLogger("google_genai.types").setLevel(logging.ERROR)

api_key = userdata.get('GEMINI')
client = genai.Client(api_key=api_key)

config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=2048),
    tools=[{"code_execution": {}}],
    max_output_tokens=65536,
)

prompt = """
Optimize the previous real-time Nonlinear Model Predictive Control (NMPC) Python implementation
to execute significantly faster while maintaining numerical stability.

The optimizations must include:
1. Vectorized NumPy matrix operations and reduced prediction horizon steps (e.g., N=15 instead of N=40)
   to eliminate redundant computational overhead.
2. Warm-starting the SQP solver using the shifted primal-dual solution from the previous timestep.
3. Profiling decorators or timing assertions within the test suite to guarantee that every
   control update step completes in under 5 milliseconds.

Run the optimized script to verify correctness and print out the execution benchmark times.
"""

print("[*] Generating optimized NMPC simulation via gemini-4-carbon...")
response = client.models.generate_content(
    model="gemini-4-carbon",
    contents=prompt,
    config=config,
)

# 1. Verify that sandbox execution succeeded
sandbox_passed = any(
    getattr(part, "code_execution_result", None) and part.code_execution_result.outcome == "OUTCOME_OK"
    for part in response.candidates[0].content.parts
)

# 2. Extract the complete, self-contained Python program from the synthesized markdown text
full_text = response.text or ""
code_blocks = re.findall(r"```python\s*(.*?)\s*```", full_text, flags=re.DOTALL)

if code_blocks:
    complete_code = max(code_blocks, key=len)
else:
    code_parts = [
        part.executable_code.code
        for part in response.candidates[0].content.parts
        if getattr(part, "executable_code", None)
    ]
    complete_code = "\n\n".join(code_parts)

output_path = Path("nmpc_trajectory_optimization_optimized.py")
output_path.write_text(complete_code, encoding="utf-8")

status = "VERIFIED (Sandbox OUTCOME_OK)" if sandbox_passed else "EXTRACTED"
print(f"[+] Status: {status}")
print(f"[+] Successfully wrote optimized NMPC script to: {output_path.resolve()}")

# 3. Execute locally
print("\n[*] Executing optimized NMPC simulation in Colab runtime:\n" + "=" * 50)
!python3 nmpc_trajectory_optimization_optimized.py

[*] Generating optimized NMPC simulation via gemini-4-carbon...
[+] Status: VERIFIED (Sandbox OUTCOME_OK)
[+] Successfully wrote optimized NMPC script to: /content/nmpc_trajectory_optimization_optimized.py

[*] Executing optimized NMPC simulation in Colab runtime:
